# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library. All dataset entities (record sets, fields, columns, etc.) are referenced by their `@id` values for clarity and reproducibility.

### Dataset Source
The dataset is defined by a Croissant schema and is available at:

[https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json](https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json)

This notebook follows the best practices for interacting with FAIR data and Croissant schemas, referencing all entities by their unique `@id` fields.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# metadata is an object, not a dict—we use its .to_json() method to view content
metadata = dataset.metadata.to_json()
print(f"Dataset: {getattr(dataset.metadata, 'name', 'unknown')}")
print(f"Description: {getattr(dataset.metadata, 'description', 'unknown')}")
print(f"Identifiers: {getattr(dataset.metadata, 'identifier', 'unknown')}")
print(f"Authors (@id): {[a['@id'] for a in metadata.get('author', [])]}")
print(f"RecordSets (@id): {metadata.get('recordSet', [])}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id` values for structured exploration.

In [ ]:
# List available record sets, their fields and columns using @id
# Because the dataset is small and the Croissant schema does not include recordSet inline, we fetch the recordSet ids

record_sets = metadata.get('recordSet', [])
if not record_sets:
    print('No recordSets found directly in metadata.')
else:
    print(f"RecordSet IDs: {[r['@id'] if isinstance(r, dict) and '@id' in r else r for r in record_sets]}")

# For demonstration, get recordSets from the schema itself
croissant_json = dataset._jsonld

rs_entities = [x for x in croissant_json if x.get('@type') in ['cr:RecordSet', 'RecordSet']]
# List each RecordSet and its Fields/Columns by @id
for rs in rs_entities:
    print(f"RecordSet: {rs.get('@id')}")
    fields = rs.get('cr:field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print("  Fields:")
    for f in fields:
        if isinstance(f, dict):
            print(f"    Field @id: {f.get('@id', str(f))}")
        else:
            print(f"    Field @id: {str(f)}")
    columns = rs.get('cr:column', [])
    if columns:
        if not isinstance(columns, list):
            columns = [columns]
        print("  Columns:")
        for col in columns:
            if isinstance(col, dict):
                print(f"    Column @id: {col.get('@id', str(col))}")
            else:
                print(f"    Column @id: {str(col)}")
    print('')
# Also show a few records from each RecordSet using its @id
for rs in rs_entities:
    rs_id = rs.get('@id')
    print(f"First 2 records from RecordSet {rs_id}:")
    try:
        records = list(dataset.records(record_set=rs_id))
        for rec in records[:2]:
            print(rec)
    except Exception as e:
        print(f"  Could not load records: {e}")
    print('---')

## 3. Data Extraction
Load data from each record set into a DataFrame for processing. All entities should be referenced by their `@id` (e.g., columns, fields).

In [ ]:
dataframes = {}
# Use all found RecordSet @id values from previous section
record_set_ids = [rs.get('@id') for rs in rs_entities]

for rs_id in record_set_ids:
    print(f"Loading records from RecordSet @id: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Columns (@id): {df.columns.tolist()}")
            print(df.head(2))
        else:
            print("No records found.")
    except Exception as e:
        print(f"Error loading records for {rs_id}: {e}")
print(f"Loaded DataFrames: {list(dataframes.keys())}")

## 4. Exploratory Data Analysis (EDA)
This section applies common EDA, such as record filtering and normalization, referencing all columns by their `@id` values.

- Remove outliers
- Normalize numeric fields
- Group data by categorical attributes


In [ ]:
# Choose a RecordSet and numeric field for demonstration
import numpy as np

# Find a RecordSet containing numeric values
chosen_rs_id = next((rs_id for rs_id, df in dataframes.items() if any(pd.api.types.is_numeric_dtype(df[c]) for c in df.columns)), None)
if not chosen_rs_id:
    raise ValueError("No RecordSet with numeric fields found.")
df = dataframes[chosen_rs_id]

# Find first numeric column (@id) in dataframe
numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if not numeric_cols:
    raise ValueError("No numeric field found in selected RecordSet.")
numeric_field_id = numeric_cols[0]

print(f"Analyzing numeric field (@id): {numeric_field_id} in RecordSet @id: {chosen_rs_id}")

threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype != object else 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records where {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized field {numeric_field_id}:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field (find string column)
group_field_id = next((col for col in df.columns if (df[col].dtype == object and col != numeric_field_id)), None)
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped data by field (@id): {group_field_id}")
    print(grouped_df.head())
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize the distribution of a numeric field and relationships between two fields, referencing column names by their `@id` values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if chosen_rs_id and numeric_field_id:
    plt.figure(figsize=(6, 4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id} (RecordSet @id: {chosen_rs_id})")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    # Use group_field_id if available for grouped visualization
    if group_field_id:
        plt.figure(figsize=(6, 4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id} (RecordSet @id: {chosen_rs_id})")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

- This notebook demonstrated how to reference and access dataset entities by their `@id` using `mlcroissant`.
- Data loading and processing steps were performed without treating `dataset.metadata` as a dictionary.
- The approach supports reproducible analysis for FAIR dataset exploration and transformation.

You can extend this workflow to other datasets and analyses by following the `@id` referencing conventions and leveraging Croissant schemas with `mlcroissant`.